In [1]:
!pip install underthesea tokenizers -q 
!pip install pandarallel -q 
!pip install pyarrow -q 
!pip install evaluate rouge_score -q
!pip install torchinfo -q 
!pip install pyngrok -q
!pip install ninja packaging --quiet
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 25.3 MB/s eta 0:00:00


In [2]:
from underthesea import word_tokenize
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
import pandas as pd
import os

# Standard library
import os
import random
import math
import copy
import multiprocessing
import time
import subprocess

# Data & utils
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

# Underthesea & tokenizers
from underthesea import word_tokenize
from pandarallel import pandarallel
from pandarallel.core import WorkerStatus
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

# PyTorch AMP
# from torch.cuda.amp import autocast, GradScaler

# Scheduler
from torch.optim.lr_scheduler import LambdaLR

# TensorBoard
from torch.utils.tensorboard import SummaryWriter

# torchinfo
from torchinfo import summary

# Ngrok & Kaggle
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient


def segment_text(text):
  try:
    return word_tokenize(text, format='text')
  except:
    return ""

if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_data_compounded_with_ner.parquet'):
  train_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_data_compounded_with_ner.parquet')
else:
  train_df = df.dropna()
  train_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train-00000-of-00001.parquet')
  train_df = train_df.dropna()
  train_df['article'] = train_df['article'].parallel_apply(segment_text)
  train_df['summary'] = train_df['summary'].parallel_apply(segment_text)
  train_df.to_parquet('train_data_compounded.parquet')
  print('Done')

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


2026-06-08 14:56:59.554967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780930619.990052      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780930620.115942      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780930621.101135      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780930621.101179      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780930621.101181      22 computation_placer.cc:177] computation placer alr

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder

if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_summarization_tokenizer.json'):
  tokenizer = Tokenizer.from_file('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_summarization_tokenizer.json')
else:
    tokenizer = Tokenizer(BPE(unk_token='<UNK>'))
    
    tokenizer.pre_tokenizer = Metaspace(replacement=" ", prepend_scheme="always")
    
    tokenizer.decoder = MetaspaceDecoder(replacement=" ", prepend_scheme="always")
    
    trainer = BpeTrainer(
        vocab_size=36000,
        min_frequency=2,
        special_tokens=[
            '<PAD>',
            '<UNK>',
            '<BOS>',
            '<EOS>'
        ]
    )
    
    text_list = train_df['article'].to_list() + train_df['summary'].to_list()
    tokenizer.train_from_iterator(text_list, trainer)
    tokenizer.save('train_summarization_tokenizer.json')
    print('Done')

In [4]:
cau_test = "Tôi đang chạy thử thuật toán BPE với cụm từ Xyz_Abc_9999 và VinFast_VF8_Pro."

encoded_test = tokenizer.encode(cau_test)
print(encoded_test.tokens)

[' Tôi', ' đang', ' chạy', ' thử', ' thuật', ' toán', ' B', 'P', 'E', ' với', ' cụm', ' từ', ' X', 'y', 'z', '_Ab', 'c_', '99', '99', ' và', ' VinFas', 't_V', 'F', '8', '_Pro', '.']


# Phải có dấu câu vì trong thực tế việc đặt dấu câu nó cũng sẽ ảnh hưởng đến ý nghĩa của câu đó.

In [5]:
import pandas as pd
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

# validate
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/val_data_compounded.parquet'):
  val_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/val_data_compounded.parquet')
else:
  val_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/valid-00000-of-00001.parquet')
  val_df = val_df.dropna()
  val_df['article'] = val_df['article'].parallel_apply(segment_text)
  val_df['summary'] = val_df['summary'].parallel_apply(segment_text)
  val_df.to_parquet('val_data_compounded.parquet')
  print('Done')

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [6]:
import torch
from torch.nn.utils.rnn import pad_sequence

val_ids = [encoding.ids for encoding in tokenizer.encode_batch(val_df['article'].to_list() + val_df['summary'].to_list())]
val_ids = pad_sequence([torch.tensor(ids) for ids in val_ids], batch_first=True, padding_value=tokenizer.token_to_id('<PAD>'))
count_unk = val_ids == tokenizer.token_to_id('<UNK>')
padding_mask = val_ids == tokenizer.token_to_id('<PAD>')
print(count_unk.sum().item())
print((count_unk.sum() / (count_unk.numel() - padding_mask.sum())).item())

5
6.806221335864393e-06


In [7]:

if train_df is not None and val_df is not None:
  train_df['article_ids'] = train_df['article'].apply(lambda x: tokenizer.encode(x).ids)
  train_df['summary_ids'] = train_df['summary'].apply(lambda x: tokenizer.encode(x).ids)
  val_df['article_ids'] = val_df['article'].apply(lambda x: tokenizer.encode(x).ids)
  val_df['summary_ids'] = val_df['summary'].apply(lambda x: tokenizer.encode(x).ids)

Vaidate thì chỉ xuất hiện 5 UNK token có nghĩa là tokenizer hoạt động tốt trên tập validate

In [8]:
# Reprocedure
import random
import numpy as np
import os

def seed(seed_value=42):
  os.environ['PYTHONHASHSEED'] = str(seed_value)
  torch.manual_seed(seed_value)
  random.seed(seed_value)
  np.random.seed(seed_value)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
  print('Done')

def seed_worker(worker_id):
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

In [9]:
from pandarallel.core import WorkerStatus
## START PROGRAM

seed(42)

from torch.utils.data import Dataset, DataLoader
import os 
import multiprocessing

num_gpus = torch.cuda.device_count()
num_cores = multiprocessing.cpu_count()

active_devices = max(1, num_gpus)

BOS = tokenizer.token_to_id('<BOS>')
EOS = tokenizer.token_to_id('<EOS>')
BATCH_SIZE = 8 * active_devices
OPTIMAL_WORKERS = num_cores
MAX_SEQ_LEN=1024
MAX_SUM_LEN=360


class SummarizationDataset(Dataset):
  def __init__(self, df):
    self.df = df
    self.has_ner = 'src_ner_mask' in df.columns and 'tgt_ner_mask' in df.columns

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    article_ids = self.df.iloc[idx]['article_ids']
    summary_ids = self.df.iloc[idx]['summary_ids']
      
    row = self.df.iloc[idx]
    
    src_ner_mask = row['src_ner_mask'] if self.has_ner else []
    tgt_ner_mask = row['tgt_ner_mask'] if self.has_ner else []
    
    return article_ids, summary_ids, src_ner_mask, tgt_ner_mask

def collate_fn(batch):
  article_tensors = []
  summary_tensors = []
  src_ner_tensors = []
  tgt_ner_tensors = []
  
  PAD_ID = tokenizer.token_to_id('<PAD>')

  for article_ids, summary_ids, src_ner, tgt_ner in batch:
    article_sliced = article_ids[:MAX_SEQ_LEN]
    article_tensors.append(torch.tensor(article_sliced))

    summary_processed = [BOS] + summary_ids[:MAX_SUM_LEN-2] + [EOS]
    summary_tensors.append(torch.tensor(summary_processed))

    if len(src_ner) > 0:
      src_ner_tensors.append(torch.tensor(src_ner[:MAX_SEQ_LEN], dtype=torch.float32))
    else:
      src_ner_tensors.append(torch.zeros(len(article_sliced), dtype=torch.float32))

    if len(tgt_ner) > 0:
      tgt_ner_processed = [0.0] + tgt_ner[:MAX_SUM_LEN-2] + [0.0]
      tgt_ner_tensors.append(torch.tensor(tgt_ner_processed, dtype=torch.float32))
    else:
      tgt_ner_tensors.append(torch.zeros(len(summary_processed), dtype=torch.float32))

  article_tensors = pad_sequence(article_tensors, batch_first=True, padding_value=PAD_ID)
  summary_tensors = pad_sequence(summary_tensors, batch_first=True, padding_value=PAD_ID)
  
  src_ner_tensors = pad_sequence(src_ner_tensors, batch_first=True, padding_value=0.0)
  tgt_ner_tensors = pad_sequence(tgt_ner_tensors, batch_first=True, padding_value=0.0)

  return article_tensors, summary_tensors, src_ner_tensors, tgt_ner_tensors


g = torch.Generator()
g.manual_seed(42)
train_dataset = SummarizationDataset(train_df)
train_dataloader = DataLoader(train_dataset, 
                              batch_size=BATCH_SIZE, 
                              shuffle=True, 
                              collate_fn=collate_fn,
                              worker_init_fn=seed_worker, 
                              generator=g, 
                              pin_memory=True, 
                              persistent_workers=(OPTIMAL_WORKERS > 0),
                              num_workers=OPTIMAL_WORKERS)

validate_dataset = SummarizationDataset(val_df)
validate_dataloader = DataLoader(validate_dataset, 
                                 batch_size=BATCH_SIZE, 
                                 shuffle=False, 
                                 collate_fn=collate_fn, 
                                 pin_memory=True, 
                                 persistent_workers=(OPTIMAL_WORKERS > 0),
                                 num_workers=OPTIMAL_WORKERS)

Done


In [10]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from torch.optim import Adam, AdamW
from torch.utils.tensorboard import SummaryWriter
from torch.amp import GradScaler, autocast 
import math
import copy
from tqdm import tqdm
import evaluate
import pandas as pd
from IPython.display import display # Dùng để in bảng đẹp trên Jupyter/Kaggle

import warnings
import transformers

# Tắt cảnh báo màu đỏ của Python
warnings.filterwarnings("ignore")

# Ép HuggingFace chỉ in ra lỗi nghiêm trọng (Tắt cái bảng LOAD REPORT đi)
transformers.logging.set_verbosity_error()

class Embeddings(nn.Module):
  def __init__(self, d_model, vocab):
    super().__init__()
    self.lut = nn.Embedding(vocab, d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.lut(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, dropout, max_len=2000):
    super().__init__()
    position = torch.arange(0, max_len).unsqueeze(-1) # max_len, 1
    pe = torch.zeros(max_len, d_model) # max_len, d_model
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.)) / d_model) # d_model,
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe.unsqueeze_(0) # 1, max_len, d_model
    self.register_buffer('pe', pe)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, step=0):
    x = x + self.pe[:, step: step + x.size(1)].requires_grad_(False)
    return self.dropout(x)

def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
  d_k = query.size(-1)
  scores = torch.matmul(query, key.transpose(-2,-1)) / math.sqrt(d_k)
  if mask is not None:
    min_value = torch.finfo(scores.dtype).min
    scores = scores.masked_fill(mask == 0, min_value)
  p_attn = F.softmax(scores, dim=-1)
  if dropout is not None:
    p_attn = dropout(p_attn)
  return torch.matmul(p_attn, value), p_attn

class MultiHeadAttention(nn.Module):
  def __init__(self, h, d_model, dropout=0.1):
    super().__init__()
    assert d_model % h == 0
    self.d_k = d_model // h
    self.h = h
    self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
    self.dropout = nn.Dropout(dropout)

  def forward(self, query, key, value, mask=None, past_key_value=None, use_cache=False, is_cross_attention=False):
    batch_size = query.size(0)
    # batch_size, h, seq_len, d_k
    if past_key_value is not None:
        if is_cross_attention: 
            query = self.linears[0](query).view(batch_size, -1, self.h, self.d_k).transpose(1,2)
            key, value = past_key_value
        else: 
            query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (query, key, value))]
            K, V = past_key_value
            key = torch.cat([K, key], dim=-2)
            value = torch.cat([V, value], dim=-2)
    else: 
        query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (query, key, value))]
        
    present_key_value = (key, value) if use_cache else None

    if mask is not None:
        mask = mask.unsqueeze(1)
    x, attn = scaled_dot_product_attention(query, key, value, mask=mask, dropout=self.dropout)
    x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)

    return self.linears[-1](x), present_key_value

class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.linear_1 = nn.Linear(d_model, d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff, d_model)
    self.activation = nn.ReLU()

  def forward(self, x):
    return self.linear_2(self.dropout(self.activation(self.linear_1(x))))


class LayerNorm(nn.Module):
  def __init__(self, d_model, eps=1e-6):
    super().__init__()
    self.a_2 = nn.Parameter(torch.ones(d_model))
    self.b_2 = nn.Parameter(torch.zeros(d_model))
    self.eps = eps

  def forward(self, x):
    x_f32 = x.float()
    mean = torch.mean(x_f32, dim=-1, keepdim=True)
    var = torch.var(x_f32, dim=-1, keepdim=True, unbiased=False)
      
    out = (x_f32 - mean) / torch.sqrt(var + self.eps)
    return (out.to(x.dtype)) * self.a_2 + self.b_2


class ResidualConnection(nn.Module):
  def __init__(self, d_model, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.layer_norm = LayerNorm(d_model)

  def forward(self, x, sublayer):
    return x + self.dropout(sublayer(self.layer_norm(x)))

def clones(module, N):
  return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class EncoderLayer(nn.Module):
  def __init__(self, d_model, d_ff, h, dropout=0.1):
    super().__init__()
    self.self_attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout=dropout)
    self.sublayer = clones(ResidualConnection(d_model, dropout=dropout), 2)
    self.d_model = d_model

  def forward(self, x, mask):
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask=mask)[0])
    x = self.sublayer[1](x, self.feed_forward)
    return x

class Encoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.layers = clones(EncoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.embedding = Embeddings(d_model, tokenizer.get_vocab_size())
    self.pe = PositionalEncoding(d_model, dropout=dropout)

  def forward(self, x, mask=None):
    x = self.embedding(x)
    x = self.pe(x)
    for layer in self.layers:
      x = layer(x, mask)
    return x

class DecoderLayer(nn.Module):
  def __init__(self, d_model, d_ff, h, dropout=0.1):
    super().__init__()
    self.masked_attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout=dropout)
    self.sublayer = clones(ResidualConnection(d_model, dropout=dropout), 3)
    self.d_model = d_model

  def forward(self, x, memory, src_mask, tgt_mask, past_kv=None, use_cache=False):
    past_self_kv = past_kv[0] if past_kv is not None else None
    past_cross_kv = past_kv[1] if past_kv is not None else None        
    present_self_kv = None
    def self_attn_wrapper(q): 
        nonlocal present_self_kv 
        out, present_self_kv = self.masked_attn(q, q, q, mask=tgt_mask, past_key_value=past_self_kv, use_cache=use_cache, is_cross_attention=False)
        return out
    x = self.sublayer[0](x, self_attn_wrapper)

    present_cross_kv = None
    def cross_attn_wrapper(q): 
        nonlocal present_cross_kv
        out, present_cross_kv = self.attn(q, memory, memory, mask=src_mask, past_key_value=past_cross_kv, use_cache=use_cache, is_cross_attention=True)
        return out
    x = self.sublayer[1](x, cross_attn_wrapper)
    x = self.sublayer[2](x, self.feed_forward)
    return x, (present_self_kv, present_cross_kv)

class Decoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.layers = clones(DecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.embedding = Embeddings(d_model, tokenizer.get_vocab_size())
    self.pe = PositionalEncoding(d_model, dropout=dropout)
    self.linear = nn.Linear(d_model, tokenizer.get_vocab_size())

  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False, step=0):
    x = self.embedding(x)
    x = self.pe(x, step=step)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs

class BaselineTransformer(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = Encoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = Decoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True, step=step)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens
              
              

In [11]:
# Count for create histogram
from tqdm import tqdm
def generate_penalty(alpha=1.0):
    _dataloader = DataLoader(train_dataset, 
                              batch_size=BATCH_SIZE, 
                              shuffle=False, 
                              collate_fn=collate_fn,
                              pin_memory=False, 
                              num_workers=0)
    VOCAB_SIZE = tokenizer.get_vocab_size()
    global_counts = torch.zeros(VOCAB_SIZE, dtype=torch.long, device='cpu')
    
    for article_ids, summary_ids, _, _ in tqdm(_dataloader): 
        text_ids = torch.cat([article_ids, summary_ids], dim=-1)
        tokens = text_ids.view(-1).cpu()
        batch_counts = torch.bincount(tokens, minlength=VOCAB_SIZE)
        global_counts += batch_counts
    
    global_counts = torch.clamp(global_counts, min=1)
    max_count = global_counts.max().float()
    
    penalty_tensor = alpha*(1.0 - (torch.log(global_counts.float()) / torch.log(max_count)))
    return penalty_tensor.unsqueeze(0) # 1, vocab_size

In [12]:
class RoPECache(nn.Module): 
    def __init__(self, d_k, dropout=0.1, max_len=2048): 
        super().__init__()
        position = torch.arange(0, max_len) # max_len,
        inv_freq = torch.exp(torch.arange(0, d_k, 2) * (-math.log(10000.)) / d_k) # d_k/2 ,
        theta = torch.einsum('i,j->ij', position, inv_freq) # max_len, d_k/2
        emb = torch.cat([theta, theta],dim=-1) # max_len, d_k
        # persistent for not save in .pt
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def forward(self, seq_len, step=0): 
        return self.cos_cached[step:step+seq_len].unsqueeze(0).unsqueeze(0), self.sin_cached[step:step+seq_len].unsqueeze(0).unsqueeze(0) # 1, 1, seq_len, d_model

def rotate_half(x): 
    x1 = x[..., :x.shape[-1]//2]
    x2 = x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin): 
    q_embed = q * cos + rotate_half(q) * sin
    k_embed = k * cos + rotate_half(k) * sin

    return q_embed, k_embed

class SwiGLU(nn.Module):
    def __init__(self, d_ff, d_model, dropout=0.1): 
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x): 
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        return self.w_down(gate * up)

class RMSNorm(nn.Module): 
    def __init__(self, d_model, eps=1e-6): 
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): 
        x_f32 = x.float()
        rms = torch.sqrt(x_f32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms.to(x.dtype)) * self.g

class ResidualConnectionWithRMS(nn.Module): 
    def __init__(self, d_model, dropout=0.1): 
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer): 
        return x + self.dropout(sublayer(self.norm(x)))
        

class MultiHeadAttentionWithRoPE(nn.Module): 
    def __init__(self, h, d_model, dropout=0.1): 
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
        self.dropout = nn.Dropout(dropout)
        self.rope_cache = RoPECache(self.d_k)

    def forward(self, q, k, v, mask=None, past_kv=None, use_cache=None, is_cross_attention=False): 
        batch_size = q.size(0)

        if past_kv is not None: 
            if (is_cross_attention): 
                query = self.linears[0](q).view(batch_size, -1, self.h, self.d_k).transpose(1,2)
                key, value = past_kv
            else: 
                query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (q, k, v))]
                seq_len = query.shape[-2]
                K, V = past_kv 
                step = K.shape[-2]
                cos, sin = self.rope_cache(seq_len, step=step)
                query, key = apply_rotary_pos_emb(query, key, cos, sin)
                key = torch.cat([K, key], dim=-2)
                value = torch.cat([V, value], dim=-2)
        else:
            # batch_size, h, seq_len, d_k
            query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (q, k, v))]
            if is_cross_attention == False: 
                seq_len = query.shape[-2]
                cos, sin = self.rope_cache(seq_len)
                query, key = apply_rotary_pos_emb(query, key, cos, sin)

        persent_kv = (key, value) if use_cache == True else None
        
        if mask is not None:
            mask = mask.unsqueeze(1)
        x, attn = scaled_dot_product_attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
    
        return self.linears[-1](x), persent_kv

class ImprovedEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.swiglu = SwiGLU(d_ff,d_model)
        self.residual = clones(ResidualConnectionWithRMS(d_model),2)

    def forward(self, x, mask): 
        x = self.residual[0](x, lambda x: self.attn(x,x,x, mask=mask)[0])
        x = self.residual[1](x, self.swiglu)
        return x

class ImprovedEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.encoder_layers = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        
    def forward(self, src_ids, mask=None): 
        x = self.embedding(src_ids) 
        for layer in self.encoder_layers:
            x = layer(x,mask)
        return x


class ImprovedDecoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.masked_attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.cross_attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.sublayer = clones(ResidualConnectionWithRMS(d_model), 3)
        self.swiglu = SwiGLU(d_ff, d_model)

    def forward(self, x, memory, src_mask, tgt_mask, past_kv=None, use_cache=False):
        past_self_kv = past_kv[0] if past_kv is not None else None
        past_cross_kv = past_kv[1] if past_kv is not None else None        
        present_self_kv = None
        def self_attn_wrapper(q): 
            nonlocal present_self_kv 
            out, present_self_kv = self.masked_attn(q, q, q, mask=tgt_mask, past_kv=past_self_kv, use_cache=use_cache, is_cross_attention=False)            
            return out
        x = self.sublayer[0](x, self_attn_wrapper)
    
        present_cross_kv = None
        def cross_attn_wrapper(q): 
            nonlocal present_cross_kv
            out, present_cross_kv = self.cross_attn(q, memory, memory, mask=src_mask, past_kv=past_cross_kv, use_cache=use_cache, is_cross_attention=True)            
            return out
        x = self.sublayer[1](x, cross_attn_wrapper)
        x = self.sublayer[2](x, self.swiglu)
        return x, (present_self_kv, present_cross_kv)

class ImprovedDecoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
    self.layers = clones(ImprovedDecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.linear = nn.Linear(d_model, tokenizer.get_vocab_size())
      
  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False):
    x = self.embedding(x)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs

class ImprovedBaselineTransformer(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = ImprovedEncoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = ImprovedDecoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [13]:
from mamba_ssm import Mamba
class BiMambaBlock(nn.Module): 
    def __init__(self, d_model, d_state=16, d_conv=4, expand=1): 
        super().__init__()
        self.mamba_forward = Mamba(
            d_model=d_model,
            d_state=d_state, # Mặc định tốt nhất
            d_conv=d_conv,   # Mặc định tốt nhất
            expand=expand    # Ép về 1 để công bằng tham số với Transformer Baseline
        )
        self.mamba_backward = Mamba(
            d_model=d_model,
            d_state=d_state, # Mặc định tốt nhất
            d_conv=d_conv,   # Mặc định tốt nhất
            expand=expand    # Ép về 1 để công bằng tham số với Transformer Baseline
        )

    def forward(self, x): 
        # batch_size, seq_len, d_model
        x_forward = self.mamba_forward(x)
        x_flipped = torch.flip(x, dims=[-2]).contiguous()
        x_backward = self.mamba_backward(x_flipped)
        x_backward = torch.flip(x_backward, dims=[-2]).contiguous()
        return x_forward, x_backward

class BiMambaEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff_new, dropout=0.1): 
        super().__init__()
        
        self.bi_mamba = BiMambaBlock(d_model=d_model, expand=1) 
        
        self.swiglu = SwiGLU(d_ff_new, d_model)
        
        self.pipe_gate = nn.Linear(d_model, 1)
        self.pipe_semantic = nn.Linear(d_model, d_model)

        self.gate_fusion = nn.Linear(2,1)  
        
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.eps = 1e-6
        

    def forward(self, x, mask, min_p=0.1): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        x_normed = self.norm(x)
        x_masked = x_normed * mask # batch_size, seq_len, d_model
        x_masked = x_masked + (1.0 - mask) * self.eps # Tránh Nan 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        gate_fwd_logits = self.pipe_gate(x_forward) # batch_size, seq_len, 1
        gate_bwd_logits = self.pipe_gate(x_backward) # batch_size, seq_len, 1

        combined_logits = torch.cat([gate_fwd_logits, gate_bwd_logits], dim=-1) # batch_size, seq_len, 2
        combined_mask = torch.sigmoid(self.gate_fusion(combined_logits)) * mask # batch_size, seq_len, 1  
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        
        combined_sem = sem_fwd + sem_bwd

        gated_sem = combined_sem * combined_mask

        final_sem = combined_sem + (gated_sem - gated_sem.detach())

        x = x + self.dropout(final_sem)

        mask_loss = (combined_mask * mask).sum() / (mask.sum() + self.eps)
        
        mask_bool = (combined_mask.squeeze(-1) > min_p) # batch_size, seq_len
        max_k = mask_bool.sum(-1).max().item() 
        max_k = max(1, max_k) 

        topk_scores, topk_indices = torch.topk(combined_mask.squeeze(-1), max_k, dim=1)
        gather_idx = topk_indices.unsqueeze(-1).expand(-1, -1, x.size(-1))
        compressed_x = torch.gather(x, dim=1, index=gather_idx) # batch_size, seq_len, d_model
        
        decoder_padding_mask = torch.gather(mask.squeeze(-1), dim=1, index=topk_indices).bool()
        decoder_padding_mask[:, 0] = True
        return compressed_x, decoder_padding_mask , mask_loss

class BiMambaEncoder(nn.Module): 
    def __init__(self, d_model, d_ff,d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.bi_mamba_block = BiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, min_p=0.1): 
        x = self.embedding(x) # batch_size, seq_len, d_model
        # mask: batch_size, 1, seq_len
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x,mask)
        output, new_mask, mask_loss = self.bi_mamba_block(x,mask, min_p = min_p)
                                                          
        new_mask = new_mask.unsqueeze(-2)  # (batch, seq_len, 1) -> (batch, 1, seq_len)
        return output, new_mask, mask_loss

class HyperSphereLinear(nn.Module): 
    def __init__(self, d_model, vocab_size, initial_gamma=10.0, tied_weight=None): 
        super().__init__()
        if tied_weight is not None:
            self.weight = tied_weight
        else:
            self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
            nn.init.xavier_uniform_(self.weight)
        self.gamma = nn.Parameter(torch.tensor(initial_gamma))
        self.eps = 1e-8
        self.register_buffer('output_scale', torch.tensor(1.0))

    def forward(self, x): 
        x_norm = F.normalize(x, p=2, dim=-1, eps=self.eps)
        w_norm = F.normalize(self.weight, p=2, dim=-1, eps=self.eps)
        gamma_clamped = torch.clamp(self.gamma, min=0.1, max=50.0)
        cosine_sim = F.linear(x_norm, w_norm)  
        logits = cosine_sim * gamma_clamped
        logits = torch.clamp(logits, min=-11.0, max=11.0)
        return logits



class HyperSphereTransformerDecoder(nn.Module): 
  def __init__(self, d_model, d_ff, h, N, dropout=0.1, tied_weight=None):
    super().__init__()
    self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
    self.layers = clones(ImprovedDecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.linear = HyperSphereLinear(d_model, tokenizer.get_vocab_size(), 
                                     tied_weight=tied_weight)     
      
  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False):
    x = self.embedding(x)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs


class TransformerWithSoftPromptMamba(nn.Module):
  def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
    super().__init__()
    self.encoder = BiMambaEncoder(d_model, d_ff,d_ff_new, h, N-1, dropout=dropout)
    self.decoder = HyperSphereTransformerDecoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory, src_mask, mask_loss = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output, mask_loss

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory, src_mask, _ = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [14]:
# 1. LAYER MỚI: Kế thừa lại toàn bộ các tầng tuyến tính của Layer cũ
class EntityGatedBiMambaEncoderLayer(BiMambaEncoderLayer): 
    def forward(self, x, mask, src_ner_mask=None, min_p=0.1): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        x_normed = self.norm(x)
        x_masked = x_normed * mask 
        x_masked = x_masked + (1.0 - mask) * self.eps 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        gate_fwd_logits = self.pipe_gate(x_forward) 
        gate_bwd_logits = self.pipe_gate(x_backward) 

        combined_logits = torch.cat([gate_fwd_logits, gate_bwd_logits], dim=-1) 
        combined_mask = torch.sigmoid(self.gate_fusion(combined_logits)) * mask   
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        
        combined_sem = sem_fwd + sem_bwd
        gated_sem = combined_sem * combined_mask
        final_sem = combined_sem + (gated_sem - gated_sem.detach())

        x = x + self.dropout(final_sem)

        base_mask_loss = (combined_mask * mask).sum() / (mask.sum() + self.eps)
        
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            ner_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='none')
            actual_ner_loss = (ner_loss * target_ner).sum() / (target_ner.sum() + self.eps)
            
            total_mask_loss = base_mask_loss + actual_ner_loss
        else:
            total_mask_loss = base_mask_loss

        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        mask_bool = (combined_mask.squeeze(-1) > min_p) 
        max_k = mask_bool.sum(-1).max().item() 
        max_k = max(1, max_k) 

        topk_scores, topk_indices = torch.topk(combined_mask.squeeze(-1), max_k, dim=1)
        gather_idx = topk_indices.unsqueeze(-1).expand(-1, -1, x.size(-1))
        compressed_x = torch.gather(x, dim=1, index=gather_idx) 
        
        decoder_padding_mask = torch.gather(mask.squeeze(-1), dim=1, index=topk_indices).bool()
        decoder_padding_mask[:, 0] = True
        
        return compressed_x, decoder_padding_mask, total_mask_loss

class EntityGatedBiMambaEncoder(BiMambaEncoder):
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__(d_model, d_ff, d_ff_new, h, N, dropout=dropout)
        self.bi_mamba_block = EntityGatedBiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None, min_p=0.1): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, new_mask, mask_loss = self.bi_mamba_block(x, mask, src_ner_mask=src_ner_mask, min_p=min_p)
                                                                    
        new_mask = new_mask.unsqueeze(-2)  
        return output, new_mask, mask_loss

class EntityGatedMambaSeq2Seq(TransformerWithSoftPromptMamba):
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__(d_model, d_ff, d_ff_new, h, N, dropout=dropout)
        self.encoder = EntityGatedBiMambaEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, src_mask, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, src_mask, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens
    

In [15]:
class TransformerWithWeightTyingAndHyperSphereNorm(nn.Module): 
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = ImprovedEncoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
    self.decoder.embedding.weight = self.encoder.embedding.weight
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [16]:
class EntityGuidedBiMambaEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff_new, dropout=0.1): 
        super().__init__()
        
        self.bi_mamba = BiMambaBlock(d_model=d_model, expand=1) 
        self.swiglu = SwiGLU(d_ff_new, d_model)
        
        self.entity_predictor = nn.Linear(d_model, 1)
        self.pipe_semantic = nn.Linear(d_model, d_model)
        
        self.prob_to_prompt = nn.Linear(1, d_model, bias=False)
        
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.eps = 1e-6
        
    def forward(self, x, mask, src_ner_mask=None, min_p=0.3): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        combined_logits = self.entity_predictor(x)
        combined_mask = torch.sigmoid(combined_logits) * mask
        
        mask_loss = torch.tensor(0.0, device=x.device)
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            mask_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='mean')
            
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        prompt_vector = self.prob_to_prompt(combined_mask)
        x_injected = x + prompt_vector
        
        x_normed = self.norm(x_injected)
        x_masked = x_normed * mask 
        x_masked = x_masked + (1.0 - mask) * self.eps 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        combined_sem = sem_fwd + sem_bwd

        output = x_injected + self.dropout(combined_sem)
        
        return output, mask_loss

class EntityGuidedBiMambaEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.bi_mamba_block = EntityGuidedBiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None, min_p=0.3): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, mask_loss = self.bi_mamba_block(x, mask, src_ner_mask=src_ner_mask, min_p=min_p)
                                          
        return output, mask_loss
        
class EntityGuidedHybridMamba(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__()
        self.encoder = EntityGuidedBiMambaEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens

In [17]:
class EntityGuidedTransformerEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.swiglu = SwiGLU(d_ff, d_model)
        self.residual = clones(ResidualConnectionWithRMS(d_model), 2)
        
        self.entity_predictor = nn.Linear(d_model, 1)
        self.prob_to_prompt = nn.Linear(1, d_model, bias=False)
        
    def forward(self, x, mask, src_ner_mask=None):
        combined_logits = self.entity_predictor(x)
        combined_mask = torch.sigmoid(combined_logits) * mask.to(x.dtype).squeeze(-2).unsqueeze(-1)
        
        mask_loss = torch.tensor(0.0, device=x.device)
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            mask_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='mean')
            
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        prompt_vector = self.prob_to_prompt(combined_mask)
        x_injected = x + prompt_vector
        
        x = self.residual[0](x_injected, lambda x: self.attn(x, x, x, mask=mask)[0])
        x = self.residual[1](x, self.swiglu)
        
        return x, mask_loss



class EntityGuidedPureTransformerEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.entity_guided_block = EntityGuidedTransformerEncoderLayer(d_model, d_ff_new,h, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, mask_loss = self.entity_guided_block(x, mask, src_ner_mask=src_ner_mask)
                                          
        return output, mask_loss
        
class EntityGuidedPureTransformer(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__()
        self.encoder = EntityGuidedPureTransformerEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens

In [18]:
def subsequence_mask(size):
  attn_shape = (1, size, size)
  return torch.tril(torch.ones(attn_shape).type(torch.bool))

class Batch:
  def __init__(self, src, tgt=None, pad_idx=0, device='cpu'):
    self.src = src.to(device)
    tgt = tgt.to(device) if tgt is not None else tgt
    self.src_mask = (self.src != pad_idx).unsqueeze(-2) # batch_size, 1, seq_len
    if tgt is not None:
      self.tgt = tgt[:, :-1]
      self.tgt_y = tgt[:, 1:]
      self.tgt_mask = self.make_std_mask(self.tgt, pad_idx).to(device)
      self.ntokens = (self.tgt_y != pad_idx).data.sum()

  @staticmethod
  def make_std_mask(tgt, pad):
    tgt_mask = (tgt != pad).unsqueeze(-2) # batch_size, 1, seq_len
    tgt_submask = subsequence_mask(tgt.size(-1)).to(tgt_mask.device) # 1, seq_len, seq_len
    return tgt_mask & tgt_submask # batch_size, seq_len, seq_len

def train_step(model, optimizer, criterion, scheduler, train_dataloader,pad_idx, epoch_num, scaler, model_name='baseline_transformer'):
  model.train()
  device = next(model.parameters()).device
  device_type = device.type
  total_loss = 0
  train_bar = tqdm(train_dataloader, desc=f'Epoch {epoch_num} [TRAIN]')
    # for src_ids, tgt_ids, src_ner_tensors, tgt_ner_tensors in train_dataloader:
  for src_ids, tgt_ids, src_ner_tensors, tgt_ner_tensors  in train_bar:
    batch = Batch(src_ids, tgt_ids, pad_idx, device=device)
    optimizer.zero_grad()
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
        if 'entity' in model_name: 
            src_ner_gpu = src_ner_tensors.to(device)
            result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask, src_ner_gpu)
        else: 
            result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        if isinstance(result, tuple):
            output, mask_loss = result
            ce_loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
            LAMBDA_SPARSITY = 0.05
            if mask_loss.dim() > 0: 
                mask_loss = mask_loss.mean()
            loss = ce_loss + LAMBDA_SPARSITY * mask_loss
        else: 
            output = result # batch_size, seq_len, vocab_size
            loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
      
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
      
    scheduler.step()
    total_loss += loss.item()
  return total_loss / len(train_dataloader)

@torch.inference_mode()
def validate_step(model, criterion, validate_dataloader, pad_idx, epoch_num, model_name='baseline_transformer'):
  model.eval()
  device = next(model.parameters()).device
  device_type = device.type
  total_loss = 0
  validate_bar = tqdm(validate_dataloader, desc=f'Epoch {epoch_num} [VALIDATE]')
  for src_ids, tgt_ids, _ , _ in validate_bar:
    batch = Batch(src_ids, tgt_ids, pad_idx, device=device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')): 
        result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        if isinstance(result, tuple):
            output, _ = result
        else:
            output = result
        loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
    total_loss += loss.item()
  return total_loss / len(validate_dataloader)

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, min_lr_ratio=0.1):
    def lr_lambda(current_step):
        # 1. Giai đoạn Warmup (Tăng dần từ 0 lên 1)
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        
        # 2. Giai đoạn Cosine Decay (Giảm dần từ 1 về min_lr_ratio)
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        
        # Công thức hàm Cosine hạ cánh mềm
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        
        # Tính toán hệ số nhân cuối cùng (đảm bảo không rớt xuống mức 0 tuyệt đối)
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

    # LambdaLR sẽ lấy `lr` ban đầu của AdamW nhân với kết quả của hàm lr_lambda ở trên
    return LambdaLR(optimizer, lr_lambda) 


def get_lr_multiplier(step_num, d_model=512, warmup_steps=4000):
    # Tránh step_num = 0 gây lỗi chia cho 0
    step_num = max(step_num, 1)
    return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (warmup_steps ** -1.5))


def train_loop(model, optimizer, criterion, scheduler, train_dataloader, val_dataloader, pad_idx, epoch, best_val_loss=None, start_epoch=0, model_name='baseline_transformer'):
  model.train()
  patience = 10
  non_improve_count = 0
  best_val_loss = float('inf') if best_val_loss is None else best_val_loss
  is_cuda = torch.cuda.is_available()
  scaler = GradScaler('cuda', enabled=is_cuda)
  writer = SummaryWriter(f'/kaggle/working/runs/{model_name}')
  for e in range(start_epoch, start_epoch + epoch):
    train_epoch_loss = train_step(model, optimizer, criterion, scheduler, train_dataloader, pad_idx, e, scaler, model_name=model_name)
    val_epoch_loss = validate_step(model, criterion, val_dataloader, pad_idx, e, model_name=model_name)
    writer.add_scalar('Loss/train', train_epoch_loss, e)
    writer.add_scalar('Loss/val', val_epoch_loss, e)

    if val_epoch_loss < best_val_loss:
      non_improve_count = 0
      best_val_loss = val_epoch_loss
      checkpoint = {
        'epoch': e,
        'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss
      }
      save_dir = f'/kaggle/working/{model_name}'
      if not os.path.exists(save_dir):
          os.makedirs(save_dir)
      torch.save(checkpoint, f'/kaggle/working/{model_name}/best_checkpoint.pt')  
    else:
      non_improve_count += 1

    if non_improve_count >= patience:
      break

  writer.close()

In [19]:
# import time
# import subprocess
# from pyngrok import ngrok
# from kaggle_secrets import UserSecretsClient

# # 1. Lấy token bảo mật
# user_secrets = UserSecretsClient()
# my_secret_token = user_secrets.get_secret("NGROK_TOKEN")
# ngrok.set_auth_token(my_secret_token)

# # Xóa các tunnel cũ nếu bạn lỡ chạy lại cell này nhiều lần (Tránh lỗi kẹt Port)
# ngrok.kill()

# # 2. Dùng subprocess để ép TensorBoard chạy ngầm (Bypass luật của Kaggle)
# print("Đang khởi động TensorBoard...")
# subprocess.Popen(['tensorboard', '--logdir', '/kaggle/working/runs', '--host', '0.0.0.0', '--port', '6006'])

# # Chờ 3 giây cho server load xong
# time.sleep(3)

# # 3. Tạo đường hầm Ngrok
# tunnel = ngrok.connect(6006)
# print("🚀 Đã xong! Bấm vào link này để xem TensorBoard:")
# print(tunnel.public_url)

    # checkpoint = {
    #     'epoch': e,
    #     'model_state_dict': model.state_dict(),
    #     'optimizer_state_dict': optimizer.state_dict(),
    #     'scheduler_state_dict': scheduler.state_dict(),
    #     'best_val_loss': best_val_loss
    #   }

In [20]:
def main_train(model, epochs, model_name='baseline_transformer'): 
    total_steps = len(train_dataloader) * epochs 

    warmup_steps = int(0.1 * total_steps) 
    
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=1.0,               
        betas=(0.9, 0.98),    
        eps=1e-5,             
        weight_decay=0.01     
    )
    # scheduler = get_cosine_schedule_with_warmup(
    #     optimizer, 
    #     num_warmup_steps=warmup_steps, 
    #     num_training_steps=total_steps,
    #     min_lr_ratio=0.1  
    # )
    scheduler = LambdaLR(optimizer, lr_lambda=lambda step: get_lr_multiplier(step, d_model=512, warmup_steps=4000))
    # Using label smoothing
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id('<PAD>'), label_smoothing=0.1)
    if os.path.exists(f'/kaggle/working/{model_name}/best_checkpoint.pt'):
        checkpoint = torch.load(f'/kaggle/working/{model_name}/best_checkpoint.pt', map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        if torch.cuda.device_count() > 1:
            print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
            model = nn.DataParallel(model)
        
        model = model.to(device)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
        remain_epoch = epochs - checkpoint['epoch'] -1 
        start_epoch=checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        
        train_loop(model, optimizer, criterion, scheduler, train_dataloader, validate_dataloader, tokenizer.token_to_id('<PAD>'), remain_epoch, 
                   best_val_loss=best_val_loss, 
                   start_epoch=start_epoch,
                  model_name=model_name)
    else: 
        train_loop(model, optimizer, criterion, scheduler, train_dataloader, validate_dataloader, tokenizer.token_to_id('<PAD>'), epochs, model_name=model_name)

In [21]:
from torchinfo import summary
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu' 
D_MODEL = 512
H = 8
N = 6
D_FF = 1365 
D_FF_NEW = 798

# 1. Khởi tạo mô hình
model_2 = ImprovedBaselineTransformer(D_MODEL, D_FF, H, N).to(device)
model_3 = TransformerWithSoftPromptMamba(D_MODEL, D_FF, D_FF_NEW, H, N).to(device)
model_4 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF, D_FF, H, N+4).to(device)
model_5 = TransformerWithWeightTyingAndHyperSphereNorm(D_MODEL, D_FF, H, N + 5).to(device)
model_6 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF, D_FF, H, N+4).to(device)
model_7 = EntityGuidedHybridMamba(D_MODEL, D_FF, D_FF, H, N+4).to(device)
model_8 = EntityGuidedPureTransformer(D_MODEL, D_FF, D_FF,H, N+5).to(device)

# 2. Wrapper thông minh của bạn (Giữ nguyên)
class ModelWrapper(nn.Module):
    def __init__(self, m): 
        super().__init__()
        self.m = m
    def forward(self, *args, **kwargs): 
        return self.m(*args, **kwargs)[0]
def count_params(model):
    # Tổng (cách torchinfo đếm - dùng để debug)
    total = sum(p.numel() for p in model.parameters())
    
    # Unique (dùng để so sánh công bằng)
    unique = sum(p.numel() for p in {p.data_ptr(): p 
                                      for p in model.parameters()}.values())
    
    print(f"Total (torchinfo style): {total:,}")
    print(f"Unique (fair comparison): {unique:,}")
    print(f"Saved by weight tying:   {total - unique:,}")

count_params(model_2)
count_params(model_3)
count_params(model_4)
count_params(model_5)
count_params(model_6)
count_params(model_7)
count_params(model_8)


# 3. Tạo dữ liệu giả (Dummy Data)
# BATCH_SIZE = 2
# SEQ_LEN = 128
# SUM_LEN = 50

# src = torch.randint(0, 100, (BATCH_SIZE, SEQ_LEN)).to(device)
# tgt = torch.randint(0, 100, (BATCH_SIZE, SUM_LEN)).to(device)
# src_mask = (src != 0).unsqueeze(-2).to(device)
# tgt_mask = Batch.make_std_mask(tgt, pad=0).to(device)

# # Dữ liệu giả cho NER mask (Chỉ dùng cho Model 4)
# src_ner_mask = torch.zeros((BATCH_SIZE, SEQ_LEN)).to(device)
# src_ner_mask[:, 10:15] = 1.0 # Giả lập có vài địa danh ở giữa câu

# ==========================================
# IN KẾT QUẢ ĐO LƯỜNG
# ==========================================

# print("\n" + "="*50)
# print("=== MODEL 2: BASELINE TRANSFORMER ===")
# print("="*50)
# summary(
#     model_2, 
#     input_data=(src, tgt, src_mask, tgt_mask),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     depth=4 # ĐÂY LÀ CHÌA KHÓA: Mở rộng tới 4 lớp con bên trong
# )

# print("\n" + "="*50)
# print("=== MODEL 3: MAMBA TOP-K (KHÔNG TIE WEIGHT) ===")
# print("="*50)
# summary(
#     ModelWrapper(model_3).to(device), 
#     input_data=(src, tgt, src_mask, tgt_mask),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     depth=4
# )

# print("\n" + "="*50)
# print("=== MODEL 4: ENTITY GATED (CÓ TIE WEIGHT) ===")
# print("="*50)
# summary(
#     ModelWrapper(model_4).to(device), 
#     # TRUYỀN THÊM src_ner_mask vào vị trí thứ 5 cho Model 4
#     input_data=(src, tgt, src_mask, tgt_mask, src_ner_mask),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     depth=4
# )

Total (torchinfo style): 99,418,272
Unique (fair comparison): 99,418,272
Saved by weight tying:   0
Total (torchinfo style): 99,418,117
Unique (fair comparison): 99,418,117
Saved by weight tying:   0
Total (torchinfo style): 92,815,877
Unique (fair comparison): 92,815,877
Saved by weight tying:   0
Total (torchinfo style): 99,256,833
Unique (fair comparison): 99,256,833
Saved by weight tying:   0
Total (torchinfo style): 92,815,877
Unique (fair comparison): 92,815,877
Saved by weight tying:   0
Total (torchinfo style): 92,816,386
Unique (fair comparison): 92,816,386
Saved by weight tying:   0
Total (torchinfo style): 99,257,858
Unique (fair comparison): 99,257,858
Saved by weight tying:   0


In [22]:
# === MODEL 2 ===

# ==============================================================================================================
# Layer (type:depth-idx)                                       Param #                   Trainable
# ==============================================================================================================
# ImprovedBaselineTransformer                                  --                        True
# ├─ImprovedEncoder: 1-1                                       --                        True
# │    └─Embedding: 2-1                                        18,432,000                True
# │    └─ModuleList: 2-2                                       --                        True
# │    │    └─ImprovedEncoderLayer: 3-1                        3,148,288                 True
# │    │    └─ImprovedEncoderLayer: 3-2                        3,148,288                 True
# │    │    └─ImprovedEncoderLayer: 3-3                        3,148,288                 True
# │    │    └─ImprovedEncoderLayer: 3-4                        3,148,288                 True
# │    │    └─ImprovedEncoderLayer: 3-5                        3,148,288                 True
# │    │    └─ImprovedEncoderLayer: 3-6                        3,148,288                 True
# ├─ImprovedDecoder: 1-2                                       --                        True
# │    └─Embedding: 2-3                                        18,432,000                True
# │    └─ModuleList: 2-4                                       --                        True
# │    │    └─ImprovedDecoderLayer: 3-7                        4,199,424                 True
# │    │    └─ImprovedDecoderLayer: 3-8                        4,199,424                 True
# │    │    └─ImprovedDecoderLayer: 3-9                        4,199,424                 True
# │    │    └─ImprovedDecoderLayer: 3-10                       4,199,424                 True
# │    │    └─ImprovedDecoderLayer: 3-11                       4,199,424                 True
# │    │    └─ImprovedDecoderLayer: 3-12                       4,199,424                 True
# │    └─Linear: 2-5                                           18,468,000                True
# ==============================================================================================================
# Total params: 99,418,272
# Trainable params: 99,418,272
# Non-trainable params: 0
# Total mult-adds (Units.MEGABYTES): 198.81
# ==============================================================================================================
# Input size (MB): 0.01
# Forward/backward pass size (MB): 158.11
# Params size (MB): 397.67
# Estimated Total Size (MB): 555.79
# ==============================================================================================================

# === MODEL 3 ===

# ===================================================================================================================
# Layer (type:depth-idx)                                            Param #                   Trainable
# ===================================================================================================================
# Model3Wrapper                                                     --                        True
# ├─TransformerWithSoftPromptMamba: 1-1                             --                        True
# │    └─BiMambaEncoder: 2-1                                        --                        True
# │    │    └─Embedding: 3-1                                        18,432,000                True
# │    │    └─ModuleList: 3-2                                       15,741,440                True
# │    │    └─BiMambaEncoderLayer: 3-3                              3,184,129                 True
# │    └─BiMambaDecoder: 2-2                                        --                        True
# │    │    └─Embedding: 3-4                                        18,432,000                True
# │    │    └─ModuleList: 3-5                                       25,196,544                True
# │    │    └─HyperSphereLinear: 3-6                                18,432,001                True
# ===================================================================================================================
# Total params: 99,418,114
# Trainable params: 99,418,114
# Non-trainable params: 0
# Total mult-adds (Units.MEGABYTES): 198.39
# ===================================================================================================================
# Input size (MB): 0.01
# Forward/backward pass size (MB): 156.96
# Params size (MB): 390.89
# Estimated Total Size (MB): 547.86
# ===================================================================================================================

In [23]:
# TRAIN
seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu' 
D_MODEL = 512
# D_FF = 2048
H = 8
N = 6
EPOCHS= 100 if device == 'cuda' else 1

# model_1 = BaselineTransformer(D_MODEL, D_FF, H, N)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_1 = nn.DataParallel(model_1)

# model_1 = model_1.to(device)
# main_train(model_1, EPOCHS, model_name='baseline_transformer')
# del model_1             # Xóa mô hình khỏi RAM CPU
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

D_FF = 1365 # 2048 * 2 / 3
# model_2 = ImprovedBaselineTransformer(D_MODEL, D_FF, H, N)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_2 = nn.DataParallel(model_2)

# model_2 = model_2.to(device)
# main_train(model_2, EPOCHS, model_name='improved_baseline_transformer')
# del model_2             # Xóa mô hình khỏi RAM CPU
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

D_FF_NEW = 798
# model_3 = TransformerWithSoftPromptMamba(D_MODEL, D_FF, D_FF_NEW, H, N)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_3 = nn.DataParallel(model_3)

# model_3 = model_3.to(device)
# main_train(model_3, EPOCHS, model_name='transformer_with_soft_prompt_mamba')
# del model_3 
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

# model_4 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF, D_FF, H, N+4)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_4 = nn.DataParallel(model_4)

# model_4 = model_4.to(device)

# main_train(model_4, EPOCHS, model_name='entity_gated_mamba_seq2Seq')
# del model_4 
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng


# model_5 = TransformerWithWeightTyingAndHyperSphereNorm(D_MODEL, D_FF, H, N + 5)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_5 = nn.DataParallel(model_5)

# model_5 = model_5.to(device)

# main_train(model_5, EPOCHS, model_name='transformer_with_hyperspherenorm_and_weight_tying')
# del model_5 
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

# model_6 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF, D_FF, H, N+4)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_6 = nn.DataParallel(model_6)

# model_6 = model_6.to(device)

# main_train(model_6, EPOCHS, model_name='entity_gated_mamba_label_smoothing')
# del model_6 
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

# model_7 = EntityGuidedHybridMamba(D_MODEL, D_FF, D_FF, H, N+4)

# if torch.cuda.device_count() > 1:
#     print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
#     model_7 = nn.DataParallel(model_7)

# model_7 = model_7.to(device)

# main_train(model_7, EPOCHS, model_name='entity_guided_hybrid_mamba')
# del model_7 
# gc.collect()            # Gọi bộ dọn rác của Python
# torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng

model_8 = EntityGuidedPureTransformer(D_MODEL, D_FF, D_FF,H, N+5).to(device)
if torch.cuda.device_count() > 1:
    print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
    model_8 = nn.DataParallel(model_8)

model_8 = model_8.to(device)

main_train(model_8, EPOCHS, model_name='entity_guided_pure_transformer')
del model_8 
gc.collect()            # Gọi bộ dọn rác của Python
torch.cuda.empty_cache()# Ép GPU nhả toàn bộ VRAM đã dùng


Done
🔥 Kích hoạt chạy song song trên 2 GPUs!


Epoch 27 [VALIDATE]: 100%|██████████| 85/85 [00:17<00:00,  4.80it/s]
